# Team Ranking Model Process

The goal is to score complete VGC teams based on their expected relative tournament performance, eventually allowing the model to assist with team building and recommendations.

## Team Representation

Each row represents **one complete team** using sparse multi-hot encoding.

Features should preserve which Pokémon owns each attribute:

```text
pokemon=Incineroar
item:Incineroar=Safety_Goggles
ability:Incineroar=Intimidate
tera:Incineroar=Ghost
move:Incineroar=Fake_Out
move:Incineroar=Parting_Shot
```

This ensures that moving an item, move, or other attribute to a different Pokémon changes the team's representation. Team and move order do not matter.

A small number of team-level features can also be added, such as:

```text
protect_count = 3
fake_out_count = 2
pair:Incineroar+Rillaboom = 1
```

Missing values should be distinguished from attributes that are not applicable to a format.

## Performance Target

Instead of predicting only Top Cut, predict normalized tournament performance:

\[
performance = 1-\frac{placement-1}{event\_size-1}
\]

This produces values from `0` to `1`, where `1` represents first place and `0` represents last place.

This preserves more information than a binary Top Cut label and allows teams to be ranked by their predicted performance.

## Model Progression

Models and features will be tested incrementally:

1. Pokémon-only features with regularized regression.
2. Add Pokémon-specific items and abilities.
3. Add moves and Tera types.
4. Add selected team-level features and common Pokémon pairs.
5. Compare regression models with a Random Forest Regressor.
6. Test a dedicated ranking model such as `XGBRanker`.

For `XGBRanker`, each tournament becomes a ranking group, allowing the model to learn which teams should rank above others within the same event.

## Training and Evaluation

Training and testing should be split **chronologically by tournament**, keeping entire events together:

```text
Earlier Events → Training
Later Events   → Testing
```

`event_id` should only be used for grouping and splitting, never as a model feature.

Models should primarily be evaluated by how well their predicted rankings match actual tournament results using metrics such as:

- Spearman rank correlation
- NDCG@K
- Actual performance of the model's highest-ranked teams

## Overall Pipeline

```text
Tournament Database
        ↓
Team Multi-Hot Encoding
        ↓
Normalized Placement Target
        ↓
Event-Based Train/Test Split
        ↓
Regression / Random Forest
        ↓
Rank-Based Evaluation
        ↓
Add Detailed Build Features
        ↓
XGBoost Ranking Model
        ↓
Team Scoring / Recommendations
```

The final score represents **historical performance associated with a team build**, not a guaranteed placement or win probability. Player skill, matchups, and tournament variance also affect results.

In [ ]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install sqlalchemy
%pip install scikit-learn
%pip install python-dotenv
%pip install psycopg2-binary

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sqlalchemy import create_engine, URL

In [ ]:
# Get connection to DB

from pathlib import Path
from dotenv import load_dotenv

#Get env varibles from FetchData folder
env_path = Path("../../Fetch_Data/.env").resolve()
load_dotenv(env_path)

# Verify correct Path
print(env_path)
print(env_path.exists())

url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "5433")),
    database=os.getenv("POSTGRES_DB")
)

engine = create_engine(url)

test_query = """
SELECT pokemon_name
FROM pokemon
"""

test_df = pd.read_sql(test_query, engine)
print(test_df.head())

In [12]:
from sqlalchemy import text

all_teams_path = Path("../../analysis/all_teams_reg.sql").resolve()

with open(all_teams_path, "r") as file:
    query = text(file.read())
    
regulation = "I"

all_teams = pd.read_sql(
    query,
    engine,
    params={'regulation' : regulation}
)

team = all_teams[all_teams["team_id"] == 3145]
features = {}

for index, row in team.iterrows():
    pokemon = row["pokemon_name"]

    # Pokemon itself
    features[f"pokemon={pokemon}"] = 1

    # Item
    if pd.notna(row["item_name"]):
        features[f"item:{pokemon}={row['item_name']}"] = 1

    # Ability
    if pd.notna(row["ability_name"]):
        features[f"ability:{pokemon}={row['ability_name']}"] = 1

    # Tera type
    if pd.notna(row["tera_type"]):
        features[f"tera:{pokemon}={row['tera_type']}"] = 1

    # Move
    if pd.notna(row["move_name"]):
        features[f"move:{pokemon}={row['move_name']}"] = 1

print(features)


def create_team_features(team_df):

    features = {}

    for _, row in team_df.iterrows():

        pokemon = row["pokemon_name"]

        # Pokemon
        features[f"pokemon={pokemon}"] = 1

        # Item
        if pd.notna(row["item_name"]):
            features[f"item:{pokemon}={row['item_name']}"] = 1

        # Ability
        if pd.notna(row["ability_name"]):
            features[f"ability:{pokemon}={row['ability_name']}"] = 1

        # Tera
        if pd.notna(row["tera_type"]):
            features[f"tera:{pokemon}={row['tera_type']}"] = 1

        # Move
        if pd.notna(row["move_name"]):
            features[f"move:{pokemon}={row['move_name']}"] = 1

    return features


{'pokemon=ursaluna': 1, 'item:ursaluna=flame-orb': 1, 'ability:ursaluna=guts': 1, 'tera:ursaluna=ghost': 1, 'move:ursaluna=headlong-rush': 1, 'move:ursaluna=facade': 1, 'move:ursaluna=earthquake': 1, 'move:ursaluna=protect': 1, 'pokemon=urshifu-rapid-strike': 1, 'item:urshifu-rapid-strike=choice-band': 1, 'ability:urshifu-rapid-strike=unseen-fist': 1, 'tera:urshifu-rapid-strike=ghost': 1, 'move:urshifu-rapid-strike=surging-strikes': 1, 'move:urshifu-rapid-strike=u-turn': 1, 'move:urshifu-rapid-strike=close-combat': 1, 'move:urshifu-rapid-strike=aqua-jet': 1, 'pokemon=miraidon': 1, 'item:miraidon=assault-vest': 1, 'ability:miraidon=hadron-engine': 1, 'tera:miraidon=electric': 1, 'move:miraidon=draco-meteor': 1, 'move:miraidon=electro-drift': 1, 'move:miraidon=snarl': 1, 'move:miraidon=volt-switch': 1, 'pokemon=incineroar': 1, 'item:incineroar=rocky-helmet': 1, 'ability:incineroar=intimidate': 1, 'tera:incineroar=bug': 1, 'move:incineroar=fake-out': 1, 'move:incineroar=parting-shot': 1, 

In [ ]:
team_feature_dicts = []

team_ids = []

for team_id, team_df in all_teams.groupby("team_id"):

    team_feature_dicts.append(
        create_team_features(team_df)
    )

    team_ids.append(team_id)
    
print(len(team_feature_dicts))
print(all_teams["team_id"].nunique())

team_metadata = (
    all_teams[
        [
            "team_id",
            "event_id",
            "event_date",
            "regulation",
            "placement",
            "event_size"
        ]
    ]
    .drop_duplicates(subset="team_id")
    .set_index("team_id")
)

y = 1 - (
    (team_metadata["placement"] - 1)
    /
    (team_metadata["event_size"] - 1)
)



1017
1017
team_id
3145    1.000000
3146    0.998652
3147    0.997305
3148    0.995957
3149    0.994609
          ...   
548     0.875862
549     0.872414
550     0.868966
551     0.865517
552     0.862069
Length: 1017, dtype: float64


In [ ]:
from sklearn.feature_extraction import DictVectorizer

vectorizer = DictVectorizer(sparse=True)

X = vectorizer.fit_transform(team_feature_dicts)

print(X.shape)

feature_names = vectorizer.get_feature_names_out()

print("Total features:", len(feature_names))

print(
    "Pokemon:",
    sum(name.startswith("pokemon=") for name in feature_names)
)

print(
    "Items:",
    sum(name.startswith("item:") for name in feature_names)
)

print(
    "Abilities:",
    sum(name.startswith("ability:") for name in feature_names)
)

print(
    "Moves:",
    sum(name.startswith("move:") for name in feature_names)
)

print(
    "Tera:",
    sum(name.startswith("tera:") for name in feature_names)
)

(1017, 1589)
Total features: 1589
Pokemon: 105
Items: 276
Abilities: 113
Moves: 777
Tera: 318
